# Data Validation with Pydantic - Practice
This notebook covers Pydantic BaseModels, type coercion, validation errors, Field constraints, custom field validators, nested models, and serialization.

### 1. Creating a BaseModel and Type Coercion
Define schemas by inheriting from Pydantic's `BaseModel`. Pydantic automatically coerces types where possible (e.g. string to integer).

In [1]:
!pip install pydantic

In [ ]:
from pydantic import BaseModel

class User(BaseModel):
    id: int
    username: str
    email: str
    is_active: bool = True

user = User(id="123", username="alice", email="alice@example.com")
print("Coerced ID:", user.id)
print("Type of ID:", type(user.id))

### 2. Validation Failures and Exception Handling
Pydantic raises a `ValidationError` when the input data cannot be parsed to match the schema types.

In [ ]:
from pydantic import BaseModel, ValidationError

class User(BaseModel):
    id: int
    username: str
    email: str

try:
    bad_user = User(id="not-an-int", username="bob", email="bob@example.com")
except ValidationError as e:
    print("Validation Error caught:")
    print(e)

### 3. Field Constraints using Field
Enforce numeric boundaries, string length constraints, and add descriptions to schema fields.

In [ ]:
from pydantic import BaseModel, Field, ValidationError

class Product(BaseModel):
    name: str = Field(min_length=2, max_length=50)
    price: float = Field(gt=0, description="Price must be greater than zero")
    stock: int = Field(default=0, ge=0)

try:
    invalid_item = Product(name="A", price=-5.0, stock=-1)
except ValidationError as e:
    print(e)

### 4. Custom Field Validators (@field_validator)
Add complex verification logic using the `@field_validator` classmethod decorator.

In [ ]:
from pydantic import BaseModel, field_validator, ValidationError

class SignUp(BaseModel):
    username: str
    password: str

    @field_validator("password")
    @classmethod
    def password_must_be_strong(cls, v: str) -> str:
        if len(v) < 8:
            raise ValueError("Password must be at least 8 characters long")
        if not any(char.isdigit() for char in v):
            raise ValueError("Password must contain at least one digit")
        return v

try:
    signup = SignUp(username="user", password="abc1")
except ValidationError as e:
    print(e)

### 5. Nested Models & Collections
Pydantic allows you to construct complex hierarchical data structures by nesting models.

In [ ]:
from pydantic import BaseModel

class Item(BaseModel):
    name: str
    price: float

class Order(BaseModel):
    order_id: str
    items: list[Item]
    tax_rate: float = 0.08
    customer_notes: str | None = None

order = Order(
    order_id="ORD123",
    items=[{"name": "Laptop", "price": 999.99}, {"name": "Mouse", "price": 29.99}]
)
print(order)

### 6. Model Serialization (dump and dump_json)
Serialize validated models into standard python dictionaries or JSON strings.

In [ ]:
from pydantic import BaseModel

class User(BaseModel):
    id: int
    username: str
    email: str

user = User(id=1, username="alice", email="alice@example.com")
print("model_dump dict:", user.model_dump())
print("model_dump_json str:", user.model_dump_json())